Author: Alana Pooler
<br>
Purpose: Complete final project

# Final Project

In this project, we will be using MLlib to build an elastic net model for predicting power consumption in tetouan city. The elastic net model will be fit using preprocessing steps applied through an MLlib pipeline, including one-hot encoding, binarization, and principle component analysis. We will also use cross validation to select optimal parameter values for the model. 

We will then use structured streaming to fit our model and generate  predictions on new data from incoming csv files. 

First, we need to import the necessary libraries and create a spark session.

In [1]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    SQLTransformer,
    Binarizer,
    StringIndexer,
    OneHotEncoder,
    VectorAssembler,
    PCA
)
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

# initialize spark session
spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/30 22:26:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/30 22:26:39 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


We need to read in the 'power_ml_data' file using pandas and convert it to a spark data frame.

We will use the Power_Zone_3 variable as our response variable and all of the other variables as predictors.

In [2]:
# read in as pandas df
pdf = pd.read_csv("data/power_ml_data.csv")

# convert to spark df and view first few rows
df = spark.createDataFrame(pdf)
df.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

Let's look at the column names and the data types of each column.

In [3]:
df.printSchema()

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)



### Part 1: Linear Regression Model

Before we can fit our model, we need to define some transformations, which we will put into a pipeline using MLlib. 

First, we need to cast the Hour column as `DoubleType` since it is currently stored as `LongType`.

We will also rename the response variable, Power_zone_3, to 'label' within the same SQLTransformer() call.

In [4]:
sql_transformer = SQLTransformer(
    statement="""
    SELECT
        *,
        CAST(Hour AS DOUBLE) AS Hour_double,
        Power_Zone_3 as label
    FROM __THIS__
    """
)

Now we need to binarize the Hour column based on the column being less than 6.5 or not, which will essentially give us an indicator of night and day.

In [5]:
hour_bin = Binarizer(
    threshold = 6.5,
    inputCol="Hour_double",
    outputCol="Hour_binary"
)

We also want to one-hot encode the Month column. First we can use StringIndexer() to map the column values to numeric indices from 0 to 11, and then we can one-hot encode the mapped values.

In [6]:
month_indexer = StringIndexer(
    inputCol="Month",
    outputCol="Month_index"
)

month_encoder = OneHotEncoder(
    inputCols=["Month_index"],
    outputCols=["Month_ohe"]
)

Next we want to run a PCA fit on the Temperature, Humidity, Wind_Speed, General_Diffuse_Flows, and Diffuse_Flows columns.

We will first use VectorAssembler() to put these variables into one column that we can use with the PCA() estimator.

In [7]:
# place variables to run PCA fit on in one column
pca_assembler = VectorAssembler(
    inputCols=[
        "Temperature",
        "Humidity",
        "Wind_Speed",
        "General_Diffuse_Flows",
        "Diffuse_Flows"
    ],
    outputCol="pca_features"
)

# run PCA
pca = PCA(
    k=2,
    inputCol="pca_features",
    outputCol="pca_output"
)

Now we can use VectorAssembler() to put all of our predictors into one 'features' column.

In [8]:
assembler = VectorAssembler(
    inputCols=[
        "pca_output",
        "Hour_binary",
        "Power_Zone_1",
        "Power_Zone_2",
        "Month_ohe"
    ],
    outputCol="features"
)

Now that we have defined all of the transformations, we can build our elastic net model using a pipeline to apply all of the transformations.

In [9]:
# define linear regression model
lr = LinearRegression(
    featuresCol="features",
    labelCol="label"
)

# build pipeline
pipeline = Pipeline(stages=[
    sql_transformer,
    hour_bin,
    month_indexer,
    month_encoder,
    pca_assembler,
    pca,
    assembler,
    lr
])

Now we need to define the parameter grid and grid values. We will test all combinations of the values 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1 for each of our two model parameters. The parameters are:
* `regParam`: controls the amount of regularization applied to the model. 
* `elasticNetParam`: controls the mix between L1 (Lasso) and L2 (Ridge) regularization.
    * A value of 0 only uses L2 regularization, a value of 1 only uses L1 regularization, and a value between 0 and 1 uses a mix of both.

In [10]:
grid_values = [0, 0.05, 0.1, 0.25, 0.5,
               0.75, 0.9, 0.95, 0.98,
               0.99, 1]

param_grid = (
    ParamGridBuilder()
    .addGrid(lr.regParam, grid_values)
    .addGrid(lr.elasticNetParam, grid_values)
    .build()
)

Next we will fit the model using 5-fold cross validation with RMSE as the criterion. Using cross validation will help us identify what values of each parameter results in the best model.

In [11]:
evaluator = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="rmse"
)

cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    parallelism=4,
    numFolds=5
)

cv_model = cv.fit(df)

26/04/30 22:27:18 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/30 22:27:18 WARN Instrumentation: [4eb234f4] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 22:27:18 WARN Instrumentation: [498671f4] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 22:27:18 WARN Instrumentation: [1790a9ea] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 22:27:18 WARN Instrumentation: [0185ad78] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 22:27:23 WARN Instrumentation: [b3cdc1af] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 22:27:23 WARN Instrumentation: [d6966563] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 22:27:23 WARN Instrumentation: [334c3fb0] regP

Now we can look at the optimal parameter values for this model, as well as the CV error, which is the average RMSE over the 5 folds.

The optimal regularization parameter is 0.1 and the optimal elastic net parameter is 0.05. Our CV error

In [12]:
# retrieve best model
best_model = cv_model.bestModel.stages[-1]

# print parameters and CV error
print("Best regParam:", best_model._java_obj.getRegParam())
print("Best elasticNetParam:", best_model._java_obj.getElasticNetParam())
print("CV Error:", min(cv_model.avgMetrics))

Best regParam: 0.25
Best elasticNetParam: 0.25
CV Error: 2147.8871281300453


Next we will find the training set RMSE by using the fitted model as a transformer and evaluating on the entire training set.

This RMSE is about the same as the cross validation error.

In [13]:
preds_cv = cv_model.transform(df)
RegressionEvaluator().evaluate(preds_cv)

2147.097324913958

The values we are predicting are large (in the tens of thousands), so the RMSE is quite large as well. Let's look at the predictions, actual values, and the residuals (observed - predicted) to get a better idea of how close the model is getting to the actual values.

The predicted values aren't perfect, but they aren't too far off either. Most of the residuals are between 1000 and 2000 higher than the actual values.

In [14]:
results = preds_cv.withColumn(
    "residual",
    col("label") - col("prediction")
)

results.select(
    "label",
    "prediction",
    "residual"
).show(10)

+-----------+------------------+------------------+
|      label|        prediction|          residual|
+-----------+------------------+------------------+
|20240.96386|20878.487809157512|-637.5239491575121|
|20131.08434| 18659.73494576147| 1471.349394238532|
|19668.43373|18204.295305419426|1464.1384245805748|
|18899.27711| 17590.24840161341|1309.0287083865878|
|18442.40964|16996.948649099904|1445.4609909000974|
|18130.12048|16517.374820839443|1612.7456591605587|
|17945.06024|16092.972090228744|1852.0881497712544|
|17459.27711|15722.452101639123|1736.8250083608764|
|17025.54217|15270.839857753163|1754.7023122468381|
|16794.21687|14938.171844510805|1856.0450254891948|
+-----------+------------------+------------------+
only showing top 10 rows


### Part 2: Streaming

Now we are going to read in a stream in the form of .csv files. First we need to setup the schema for the stream and the readStream.

In [15]:
schema = df.schema

stream_df = (
    spark.readStream
    .schema(schema)
    .option("header", True)
    .csv("stream_output/")
)

Now, we will use our stream and the model transformer we defined to get predictions from the incoming data. We will create a residual column with these predictions like we did in part 1.

In [16]:
pred_stream = cv_model.transform(stream_df)

pred_stream = pred_stream.withColumn(
    "residual",
    col("label") - col("prediction")
)

pred_stream = pred_stream.select(
    "label",
    "prediction",
    "residual"
)

Next we will use another transformation on the stream to rename the response variable to 'label'. Then we will join this stream with the stream we made above.

In [17]:
label_stream = stream_df.withColumnRenamed(
    "Power_Zone_3",
    "label"
)

joined_stream = pred_stream.join(
    label_stream,
    on="label",
    how ="inner"
)

Lastly, we will write the stream to the console using the `append` output mode and start the query.

After running this cell, we will use the console to run our data_production.py file. This will take a random sample of 5 rows from 'power_streaming_data.csv' and write it to a csv file in the '/stream_output' folder. Then the stream will read in that csv file containing the random sample, fit the elastic net model, and return a data frame containing the label, prediction, residual, and the predictor columns. This will be repeated 15 times on different random samples of data from 'power_streaming_data.csv'.

In [19]:
query = (
    joined_stream.writeStream
    .outputMode("append")
    .format("console")
    .start()
)

26/04/30 22:42:51 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-41525d21-cd89-4499-a6ce-3237b30149d9. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/30 22:42:51 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|20154.21687| 19870.10369848741|284.11317151258845|      19.08|    73.9|     0.071|                0.113|        0.048| 39095.38462| 32065.28926|   11|  19|
|16886.74699| 19576.63942841031|-2689.892438410312|      15.82|    72.0|     4.911|                119.2|        117.7| 34869.87342| 21330.09119|    1|  12|
|26247.87692|22193.995800877034| 4053.881119122965|      22.39|    76.9|     4.919|                0.084|        0.107

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|25697.12563|  25437.6293459935| 259.4962840064982|      13.12|    75.9|     0.074|                0.081|        0.145| 43114.57627| 25754.40729|    2|  19|
|16362.58065|15611.364107735448| 751.2165422645521|       11.2|    87.5|     0.075|                0.066|        0.134| 26410.21277| 15095.12195|    3|   1|
|17181.68675| 19332.87051065959|-2151.183760659591|      19.93|   43.76|     0.087|                0.088|        0.067

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
| 12990.6383|14210.532198203291|-1219.8938982032905|      22.64|   68.37|      4.92|                136.5|         94.9| 35316.23632| 22604.56432|   10|  14|
|30553.30544|27588.338580049283| 2964.9668599507168|      27.95|    71.8|     4.916|                308.0|        237.5| 36760.66445| 22682.27848|    7|  14|
|26251.38075|26970.632795620022| -719.2520456200218|      22.49|   62.46|     4.906|                0.084|       

-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|15919.59799| 15595.40226219158|   324.195727808421|       16.1|   41.69|     0.086|                278.9|        305.1| 30746.44068| 17686.32219|    2|  17|
|22013.92713| 21627.05945726277| 386.86767273722944|       17.4|   67.93|     4.922|                0.059|        0.096| 37619.40984| 23829.10217|    5|  23|
|15339.91632|21637.961660132623| -6298.045340132623|      24.29|    82.2|     4.915|                0.088|       

...

-------------------------------------------
Batch: 13
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|17638.55422| 19263.87661013147|-1625.3223901314668|       17.8|   65.35|      4.92|                222.8|        192.3| 35210.12658|  20600.6079|    1|  13|
|13465.16129|12747.351970034293|  717.8093199657069|      11.14|    87.9|     0.095|                0.044|        0.145| 21961.53191| 13514.63415|    3|   4|
|19357.09091|18955.548616203494| 401.54229379650496|      13.26|   69.47|     0.083|                0.051|      

-------------------------------------------
Batch: 14
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|16679.87743|16883.719593209833| -203.8421632098325|       21.0|    78.2|     4.917|                0.091|        0.093| 36401.41593| 20881.49688|    9|  22|
|26383.75385|27345.263331433744| -961.5094814337426|      19.18|    78.5|     0.071|                0.048|        0.111| 41591.52318| 25656.54886|    6|   2|
|13905.04523|12676.757716765387| 1228.2875132346126|      13.73|   69.27|     0.073|                0.062|      

-------------------------------------------
Batch: 15
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|18964.58291|17082.852946287527| 1881.7299637124743|      13.24|    71.0|     0.083|                0.048|        0.141| 30593.89831| 18576.29179|    2|  23|
|20746.83386|21848.689294267824|-1101.8554342678253|      21.93|   65.56|     4.912|                 0.08|        0.115| 29490.43285| 19478.77508|    8|   4|
|8061.464586| 8490.745430631428| -429.2808446314284|       6.87|    87.5|     0.086|                103.6|      

-------------------------------------------
Batch: 16
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|14291.15424| 15229.62893862533|-938.4746986253303|      21.97|    86.1|     4.914|                0.066|        0.104|  31132.0354| 18827.02703|    9|   0|
|15950.06002|18392.477105515598|-2442.417085515597|      12.24|    86.3|     0.076|                 0.04|        0.126| 39628.89734| 33974.83891|   12|  20|
|14353.45455|17042.609873709695|-2689.155323709694|      14.77|    84.8|     0.069|                 85.3|         79.

-------------------------------------------
Batch: 17
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|12781.98995|15202.141926125696|-2420.151976125697|       14.0|   67.56|     0.084|                34.71|        33.15| 27890.84746| 17894.22492|    2|   8|
|19241.35385|17668.166909150128|1573.1869408498715|      24.42|   46.51|     0.081|                668.1|        192.5| 32932.45033| 19485.65489|    6|  16|
|17135.27638|16944.083187350992|191.19319264900696|      13.84|    72.4|     0.073|                146.1|        145.